# Integration pilot: creation events, metadata declarations, uncertainty

**Joint responsibility:** Claire and Shilin. **Independent joint review:** pending. This editable Colab joins Claire's pinned real on-chain records to Shilin's real off-chain pilot through exact event IDs and evidence. It starts from a fixed public snapshot in a fresh CPU runtime; it preserves unmatched and access-restricted events.

[Integration proposal and diagrams](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/PROPOSAL.md) · [Shilin code/data](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1) · [Claire pinned input](https://huggingface.co/datasets/global-nomad-nexus/claire-threechain-v1/tree/8b29598a6565b67a8a943962dbf77f3d6b2559de) · [Dictionary](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/DATA_DICTIONARY.md). This is an engineering checkpoint for a possible *Scientific Data* descriptor, not a population estimate.

<a id="part-1"></a>
## Part 1 — Overview and navigation

The integration unit is one committed decoded token creation event, identified by `launch_record_id`. Each event is retained, even if no off-chain request was possible. A linked URL is an observed declaration with two evidence paths: the creation event supplied a URI, and a later response contained the URL field. We do not infer ownership or creation-time site state.

1. [Overview](#part-1)  2. [Component releases](#part-2)  3. [Acquire fixed inputs](#part-3)  4. [Inspect compatibility](#part-4)  5. [Join and validate](#part-5)  6. [Inspect links and unknowns](#part-6)  7. [Descriptive reuse](#part-7)  8. [Technical validation and handoff](#part-8).

<a id="part-2"></a>
## Part 2 — Two real component releases and the linkage contract

Claire's versioned [on-chain release](https://huggingface.co/datasets/global-nomad-nexus/claire-threechain-v1/tree/8b29598a6565b67a8a943962dbf77f3d6b2559de) provides three-chain raw and decoded observations for **2026-09-14 12:00:00–12:05:00 UTC**. The event-level selection implemented by [build_cohort.py](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/build_cohort.py) gives 61 Pump.fun and 55 Four.meme committed creations, with zero recognized Clanker creations. `creation_source_record_id`, `creation_raw_ref`, decoder version and Claire revision remain in the cohort. Shilin's [fixed release](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/release) contains response snapshots, declarations, exact candidates, evidence, typed assertions and a total coverage ledger.

**Join contract:** `launch_record_id` joins cohort to coverage and candidates; `object_id` is a chain-qualified token; `candidate_id` joins to evidence and assertion; `snapshot_id` joins declarations to the retrieved response. Exact on-chain URI is accepted as a direct declaration. A JSON field URL is accepted only as a claim in that response. Similar names and symbols never auto-link. `chain_event_time_utc` dates creation; `retrieved_at_utc` dates our observation; source-claimed publication dates are not verified and are not silently substituted. `as_of_eligibility=unknown` on response-derived links prevents retrospective leakage.

<img src="https://raw.githubusercontent.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/figures/dgp.svg" width="900" alt="Data generating process diagram" />

```mermaid
flowchart LR
 A[Creator/platform] --> B[Chain creation event]
 A -. contingent .-> C[Metadata URI and JSON]
 B --> D[Creation cohort]
 C --> E[Timed response snapshot]
 D --> F[Exact evidence-backed assertion]
 E --> F
```

The dotted path can fail or change over time. Full editable captions and stage descriptions are in the proposal.

In [ ]:
import os, sys, json, hashlib, tarfile, tempfile, urllib.request
from collections import Counter
from pathlib import Path
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow==25.0.1"])
    import pyarrow as pa
    import pyarrow.parquet as pq

CODE_COMMIT = "1fec501de1de09d9cc2b9c69ce350338a40af889"
TESTED_ARROW = "25.0.1"

local = os.environ.get("PILOT_LOCAL_REPO")
if local:
    ROOT = Path(local).expanduser().resolve()
    print("Local test checkout:", ROOT)
else:
    url = f"https://codeload.github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/tar.gz/{CODE_COMMIT}"
    request = urllib.request.Request(url, headers={"User-Agent": "shilin-offchain-pilot-colab/1.0"})
    body = urllib.request.urlopen(request, timeout=90).read()
    tmp = Path(tempfile.mkdtemp(prefix="shilin-pilot-"))
    archive = tmp / "repo.tar.gz"
    archive.write_bytes(body)
    with tarfile.open(archive, "r:gz") as tar:
        names = tar.getnames()
        prefix = names[0].split("/")[0]
        tar.extractall(tmp, filter="data")
    ROOT = tmp / prefix
    print("Downloaded pinned code and release:", CODE_COMMIT)

PILOT = ROOT / "pilots" / "shilin-offchain-v1"
RELEASE = PILOT / "release"
sys.path.insert(0, str(PILOT))
from verify_release import verify
verification = verify(RELEASE)
assert verification["passed"], verification
manifest = json.loads((RELEASE / "release_manifest.json").read_text())
print("Fixed release verified:", verification)
print("PyArrow runtime:", pa.__version__, "; locally tested:", TESTED_ARROW)
print("Claire input revision:", manifest["claire_hf_revision"])

<a id="part-3"></a>
## Part 3 — Acquire the fixed, versioned inputs

The setup downloads an immutable GitHub commit, verifies every release file size and SHA-256 against `release_manifest.json`, then reads seven Parquet tables. The source collection itself used real HTTP requests and Claire's pinned Parquet; this notebook deliberately starts from the released snapshot so a future reader can reproduce the join after websites change. A current live request is demonstrated in the companion off-chain notebook.

In [ ]:
TABLES = {name: pq.read_table(RELEASE / (name + ".parquet")).to_pylist() for name in ("onchain_launch_cohort","offchain_snapshots","offchain_declarations","linkage_candidates","linkage_evidence","linkage_assertions","coverage_ledger")}
print("Rows by table:", {name:len(rows) for name,rows in TABLES.items()})
print("Pinned Claire revision:", manifest["claire_hf_revision"])
print("Sample creation source:", {k:TABLES["onchain_launch_cohort"][0][k] for k in ("launch_record_id","object_id","creation_raw_ref","chain_event_time_utc")})

<a id="part-4"></a>
## Part 4 — Inspect compatibility and record counts

Before joining, inspect units, uniqueness, source revision and missing keys. `launch_record_id` is an event key, not a token symbol. Four.meme has no exact metadata URI in the decoded event; three exact-address API probes returned 403 and 52 remain unattempted. Pump.fun has 61 event URIs but 57 distinct URI values. The difference is deduplication of requests, not disappearance of events.

In [ ]:
cohort = TABLES["onchain_launch_cohort"]
coverage = TABLES["coverage_ledger"]
assert len(cohort) == len(coverage) == 116
assert len({x["launch_record_id"] for x in cohort}) == 116
assert {x["launch_record_id"] for x in coverage} == {x["launch_record_id"] for x in cohort}
assert {x["claire_hf_revision"] for x in cohort} == {manifest["claire_hf_revision"]}
print("By platform:", dict(Counter(x["platform_id"] for x in cohort)))
print("Unique tokens:", len({x["object_id"] for x in cohort}))
print("Coverage:", dict(Counter(x["coverage_state"] for x in coverage)))
print("Nonempty/distinct metadata URIs:", sum(bool(x["metadata_uri_declared"]) for x in cohort), len({x["metadata_uri_declared"] for x in cohort if x["metadata_uri_declared"]}))

<a id="part-5"></a>
## Part 5 — Process the exact join and preserve uncertainty

The left join keeps all 116 cohort events. Assertions are one-to-many: a token can have its on-chain URI assertion and several JSON field URL assertions. We therefore do not count assertion rows as tokens. The [processing implementation](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/process_linkage.py) creates candidates and evidence and the [validation code](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/validate_pilot.py) checks foreign keys and time. The diagram can be edited in the [proposal](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/PROPOSAL.md).

<img src="https://raw.githubusercontent.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/figures/pipeline.svg" width="1000" alt="Integration pipeline" />

```mermaid
flowchart LR
 A[Claire creation cohort] --> C[Exact event URI]
 B[Shilin response snapshots] --> D[JSON field URLs]
 C --> E[Candidate and evidence]
 D --> E
 E --> F[Typed assertions]
 F --> G[116-row coverage ledger]
```

Any URI failure, absent field, 403, or unattempted BSC address stays separate. A `no_declaration` label means no covered URL field in a successful JSON response; it does not imply no online presence.

In [ ]:
by_id = {x["launch_record_id"]:x for x in coverage}
joined = [{**x, "coverage_state":by_id[x["launch_record_id"]]["coverage_state"], "snapshot_id":by_id[x["launch_record_id"]]["snapshot_id"]} for x in cohort]
assert len(joined) == 116 and len({x["launch_record_id"] for x in joined}) == 116
assert sum(x["coverage_state"]=="declaration_observed" for x in joined)==29
assert sum(x["coverage_state"]=="no_declaration" for x in joined)==32
assert sum(x["coverage_state"]=="access_restricted_here" for x in joined)==3
assert sum(x["coverage_state"]=="not_attempted" for x in joined)==52
print("Joined event rows:", len(joined), "state counts:", dict(Counter(x["coverage_state"] for x in joined)))

<a id="part-6"></a>
## Part 6 — Inspect positive, unmatched and time-uncertain cases

The Morfik case has an exact on-chain URI and a later JSON `/website` value. Another JSON places an X post URL in a `website` field, so the released type remains `declares_website_field_url` with `target_class=social_post`. A third successful JSON has no covered URL field. Three Four.meme exact-address probes are `access_restricted_here`; 52 have `not_attempted`. No BSC positive link is fabricated. [Manual spot review](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/REVIEW_EXAMPLES.md).

In [ ]:
assertions = TABLES["linkage_assertions"]
evidence = TABLES["linkage_evidence"]
declarations = TABLES["offchain_declarations"]
examples = {kind:next(x for x in joined if x["coverage_state"]==kind) for kind in ("declaration_observed","no_declaration","access_restricted_here","not_attempted")}
for kind,row in examples.items():
    print(kind, row["launch_record_id"], row["object_id"], "snapshot", row["snapshot_id"])
morfik = next(x for x in assertions if x["assertion_id"] == "assert:136781253019bac2c496113b")
print("Morfik evidence:", morfik["right_value"], morfik["evidence_ids"], morfik["as_of_eligibility"])
print("Morfik evidence types:", [(x["evidence_kind"],x["field_pointer"]) for x in evidence if x["evidence_id"] in morfik["evidence_ids"]])
print("Website-field social posts:", sum(x["field_name"]=="website" and x["target_class"]=="social_post" for x in declarations))
assert all(x["as_of_eligibility"]=="unknown" for x in assertions if x["relation_type"]!="declares_metadata_uri")

<a id="part-7"></a>
## Part 7 — Descriptive example for research reuse

The following table counts event-level coverage by chain/platform. It is useful for understanding where this pilot has link evidence and where a source blocks access. It is not a success rate or statement about the projects. The 39 declaration rows come from 29 Pump.fun token events; 100 assertion rows include 61 direct URI assertions. Keep denominators and units explicit.

[DIVE](https://doi.org/10.1038/s41597-026-07025-5) models transparent data construction and technical validation; [Multi-Chain Graphs of Graphs](https://proceedings.neurips.cc/paper_files/paper/2024/file/3205b048f9cc54b9f7963db0b0f52d53-Paper-Datasets_and_Benchmarks_Track.pdf) §3.2 motivates a clear chain-specific frame. Our five-minute slice cannot stand in for their longer histories.

In [ ]:
summary = {}
for platform in sorted({x["platform_id"] for x in joined}):
    rows = [x for x in joined if x["platform_id"]==platform]
    summary[platform] = {"creation_events":len(rows), "coverage_states":dict(Counter(x["coverage_state"] for x in rows))}
print(json.dumps(summary, indent=2))
print("Field URL declarations:", len(declarations), "typed assertions:", len(assertions))
assert len(declarations)==39 and len(assertions)==100

<a id="part-8"></a>
## Part 8 — Validation, interpretation and handoff

The fixed verifier checks every public file hash and row count, event and coverage keys, evidence foreign keys and temporal flags. Original source-side checks also rehash 94 local HTTP response bodies and re-decode three Claire raw Pump events; the [actual report](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/release/validation_report.md) records 13 passing checks. The released raw response bytes are withheld pending redistribution review, so offline readers can verify the released join but must re-fetch live metadata for an independent parse. Current source data can change. CIDv0 DAG-PB content was not cryptographically checked. Independent Claire review and joint notebook execution must be recorded by both authors before telling the supervisor that cross-reproduction is complete.

The larger *Scientific Data* Data Descriptor still needs a justified publication cohort, source-rights resolution, archival identifier and broader technical validation. This pilot establishes a bounded, inspectable workflow.

In [ ]:
result = verify(RELEASE)
print(json.dumps(result, indent=2))
assert result["passed"] and result["coverage_states"] == {"access_restricted_here":3,"not_attempted":52,"no_declaration":32,"declaration_observed":29}
print("Completed fixed-snapshot integration tutorial; coauthor review remains a separate recorded step.")